# Submissão A - DNN em NumPy
## Gera o ficheiro CSV para a 1.ª submissão
### Treino interno

In [2]:
import pickle
import pandas as pd
import numpy as np
import re
from collections import Counter
import sys
import os

sys.path.append("../src")
sys.path.append("../models")

with open("../models/modelo_numpy_submA.pkl", "rb") as f:
    artefactos = pickle.load(f)

model_binary = artefactos["modelo_binario"]
model_llm = artefactos["modelo_llm"]
vocab = artefactos["vocabulario"]
idf = artefactos["idf"]
llm_mapping = artefactos["llm_mapping"]

print("Modelo carregado com sucesso.")

Modelo carregado com sucesso.


In [3]:
import re
from collections import Counter

STOPWORDS = {
    "the", "a", "an", "and", "or", "of", "to", "in", "on", "at", "for", "with",
    "is", "are", "was", "were", "be", "been", "being", "this", "that", "these",
    "those", "it", "its", "as", "by", "from", "but", "about", "into", "than",
    "then", "so", "such", "if", "their", "there", "they", "them", "he", "she",
    "you", "your", "we", "our", "i", "my", "me", "his", "her", "what", "which",
    "who", "whom", "can", "could", "should", "would", "do", "does", "did", "have",
    "has", "had", "not", "no", "yes", "will", "just"
}

def tokenize(text):
    text = text.lower()
    tokens = re.findall(r"\b[a-zA-ZÀ-ÿ]{2,}\b", text)
    return [tok for tok in tokens if tok not in STOPWORDS]

def build_vocabulary(texts, max_features=5000):
    counter = Counter()
    for text in texts:
        counter.update(tokenize(text))
    most_common = counter.most_common(max_features)
    vocab = {word: idx for idx, (word, _) in enumerate(most_common)}
    return vocab

def compute_idf(tokenized_texts, vocab):
    n_docs = len(tokenized_texts)
    df_counts = np.zeros(len(vocab), dtype=np.float64)

    for tokens in tokenized_texts:
        unique_tokens = set(tok for tok in tokens if tok in vocab)
        for tok in unique_tokens:
            df_counts[vocab[tok]] += 1

    idf = np.log((1 + n_docs) / (1 + df_counts)) + 1
    return idf

def vectorize_tfidf(texts, vocab, idf):
    X = np.zeros((len(texts), len(vocab)), dtype=np.float64)

    for i, text in enumerate(texts):
        tokens = [tok for tok in tokenize(text) if tok in vocab]
        if not tokens:
            continue

        counts = Counter(tokens)
        total_terms = len(tokens)

        for tok, count in counts.items():
            j = vocab[tok]
            tf = count / total_terms
            X[i, j] = tf * idf[j]

    return X

def l2_normalize_rows(X, eps=1e-12):
    norms = np.linalg.norm(X, axis=1, keepdims=True)
    return X / (norms + eps)


class SimpleDataset:
    def __init__(self, X, y):
        self.X = X
        self.y = y

In [7]:
df_subm1 = pd.read_csv("../data/subm1.csv", sep=None, engine="python")

print(df_subm1.shape)
df_subm1.head()

(150, 2)


,﻿ID,Text
0,D2-1,A covalent bond is a chemical bond that involv...
1,D2-2,A covalent bond forms when two atoms share one...
2,D2-3,A covalent bond is a type of chemical bond whe...
3,D2-4,A covalent bond is a chemical bond that involv...
4,D2-5,Driven by exciting developments in the field o...


In [8]:
X_subm1_text = df_subm1["Text"].fillna("").astype(str).values
X_subm1 = vectorize_tfidf(X_subm1_text, vocab, idf)
X_subm1 = l2_normalize_rows(X_subm1)

dummy = SimpleDataset(X_subm1, np.zeros(len(X_subm1), dtype=int))

print(X_subm1.shape)

(150, 5000)


In [16]:
prob_bin = model_binary.predict(dummy)
pred_bin = (prob_bin >= 0.512).astype(int).flatten()

print("Humans:", (pred_bin == 0).sum())
print("LLM:", (pred_bin == 1).sum())

Humans: 51
LLM: 99


In [10]:
indices_llm = np.where(pred_bin == 1)[0]
X_llm = X_subm1[indices_llm]

if len(indices_llm) > 0:
    dummy_llm = SimpleDataset(X_llm, np.zeros(len(X_llm), dtype=int))
    prob_llm = model_llm.predict(dummy_llm)
    pred_llm = np.argmax(prob_llm, axis=1)
else:
    pred_llm = np.array([], dtype=int)

In [11]:
final_labels = np.array(["Human"] * len(df_subm1), dtype=object)

for pos, pred in zip(indices_llm, pred_llm):
    final_labels[pos] = llm_mapping[int(pred)]

pd.Series(final_labels).value_counts()

Human        51
Google       45
Anthropic    41
OpenAI       12
Meta          1
Name: count, dtype: int64

In [12]:
df_output = df_subm1.copy()
df_output["Labels"] = final_labels

df_output.to_csv("subm1-g4-MIA-A.csv", index=False)

print("CSV criado com sucesso.")
df_output.head()

CSV criado com sucesso.


,﻿ID,Text,Labels
0,D2-1,A covalent bond is a chemical bond that involv...,Anthropic
1,D2-2,A covalent bond forms when two atoms share one...,Human
2,D2-3,A covalent bond is a type of chemical bond whe...,Human
3,D2-4,A covalent bond is a chemical bond that involv...,OpenAI
4,D2-5,Driven by exciting developments in the field o...,Human
